In [1]:
%load_ext autoreload
%autoreload 2

import os

os.chdir("../../")

### Split the human annotated data into train/test

- Run `python -m src.data.restructure_entrainment_coding_data --output_csv` first.

In [2]:
import pandas as pd

dogs_df = pd.read_excel("baskets_dogs_data/dogs.xlsx")
baskets_df = pd.read_excel("baskets_dogs_data/baskets.xlsx")
pairs = dogs_df.columns.tolist()[3:]
pairs

['Pair_20',
 'Pair_21',
 'Pair_22',
 'Pair_23',
 'Pair_24',
 'Pair_34',
 'Pair_35',
 'Pair_37',
 'Pair_43',
 'Pair_45']

In [3]:
from glob import glob


annotated_df_fp = "baskets_dogs_data/entrainment_coding_annotated_data.csv"

if os.path.exists(annotated_df_fp):
    annotated_df = pd.read_csv(annotated_df_fp)
    print(f"Loaded existing annotated data with {len(annotated_df)} rows.")
else:
    annotated_data = []
    pairs_ixes = set(p.split("_")[-1] for p in pairs)
    csv_files = glob("baskets_dogs_data/restructured_entrainment_coding_data/**/*.csv", recursive=True)

    for csv_file in csv_files:
        parts = csv_file.split("/")
        object_category = parts[2]
        pair_ix = parts[3].replace("pair", "").replace(".csv", "")

        df = pd.read_csv(csv_file)
        df.insert(0, "object_category", object_category)
        df.insert(1, "pair_ix", pair_ix)

        split = "train" if pair_ix not in pairs_ixes else "test"
        df.insert(2, "split", split)
        annotated_data.append(df)

    annotated_df = pd.concat(annotated_data).reset_index(drop=True)
    annotated_df.to_csv(annotated_df_fp, index=False)
    print(f"Saved annotated data with {len(annotated_df)} rows to {annotated_df_fp}.")

annotated_df.head()

Loaded existing annotated data with 600 rows.


,object_category,pair_ix,split,FirstRoundCardIndex,Round1,Round2,Round3,Round4
0,baskets,45,test,Card1,"rectangle shaped, handle in middle, no lid, empty","rectangle shaped with handle in middle, no lid",rectangle shaped with handle in middle,rectangular shaped with handle in middle
1,baskets,45,test,Card2,"Easter basket, pink things, pink wood, twisted...",Easter basket,Easter basket,Easter basket
2,baskets,45,test,Card3,"circular, criss-cross box, handle, one of them...",circular with criss-crosses and little black s...,"confusing one that's circular, with criss-cros...",confusing one with black stuff inside
3,baskets,45,test,Card4,"doesn't have a lid, wide, circle, really tiny ...","widest, bowl shaped, no handles and no lid","most circular and widest, with no lid and no h...","wide bowl, biggest, most circular"
4,baskets,45,test,Card5,"pink ribbon going around it, small, criss-cros...","smallest with pink ribbon going around it, no lid","smallest, pink ribbon going around it",smallest with pink ribbon going around


In [4]:
train_df = annotated_df[annotated_df.split == "train"]
test_df = annotated_df[annotated_df.split == "test"]

### Few-shot exemplars

- 2-shots. One for dogs, the other for baskets.

In [5]:
from random import sample

random_pair_ixes = sample(train_df.pair_ix.unique().tolist(), 2)
random_objects = sample(["baskets", "dogs"], 2)
random_pairs = list(zip(random_pair_ixes, random_objects))
random_pairs

[(17, 'baskets'), (18, 'dogs')]

In [6]:
import json

exemplars = []

for pair_ix, object_category in random_pairs:
    df_subset = train_df[
        (train_df.pair_ix == pair_ix)
        & (train_df.object_category == object_category)
    ]
    random_round = sample(["Round1", "Round2"], 1)[0] # choose from more elaborate rounds
    print(f"Pair {pair_ix} | Object: {object_category} | Round: {random_round}")
    extracted_descriptions = df_subset[random_round]
    exemplar = dict()

    for i, extracted_description in enumerate(extracted_descriptions):
        exemplar[f"object_#{i+1}"] = extracted_description
    
    exemplars.append(json.dumps(exemplar, indent=4))
    
for exemplar in exemplars:
    print(exemplar)

Pair 17 | Object: baskets | Round: Round1
Pair 18 | Object: dogs | Round: Round1
{
    "object_#1": "long, handle, narrow",
    "object_#2": "handle, pretty small",
    "object_#3": "shorter handle, full, lot of spaces in between with the weaving",
    "object_#4": "does not have handle, pretty big",
    "object_#5": "small, does not have handle, spaces in between weaving",
    "object_#6": "very light in color, does not have handle, circular base that's apart from basket, no cover, little stand at bottom, green background thingy",
    "object_#7": "cover, two handles at edge",
    "object_#8": "v-shaped",
    "object_#9": "handle, cover",
    "object_#10": "handle, cover, little bit longer than previous"
}
{
    "object_#1": "mainly black on stomach, mid-part of body is black in color, brown legs",
    "object_#2": "face is facing left side",
    "object_#3": "resembles Labrador retriever, tail elongated, posing in left position, face is facing left side, yellowish beige, collar, thin

## Prompting

- This is essentially an extractive summarization task

In [7]:
from string import Template

prompt_temp = f"""
This is an extractive summarization task. You will be provided with a transcript \
of a conversation between two participants engaged in a collaborative object matching task. \
One partcipant is responsible for describing target objects one at a time, \
while the other participant is responsible for matching the described objects. \
Your task is to extract the descriptive phrases used by the describer \
during the conversation for each target object descrived. Extract these \
phrases verbatim from the excerpts (exclude disfluencies and fillers) and output them in a JSON fomat as follows:

{{
    "object_#1": "descriptive phrase for object 1",
    "object_#2": "descriptive phrase for object 2",
    ...
}}

Here are two examples to illustrate the format:

Exemplar 1:

{exemplars[0]}

Exemplar 2:

{exemplars[1]}

Now, please extract the descriptive phrases for each target object in order from the transcript. \
Do not output any additional text other than the JSON object.

$transcript
""".strip()

prompt_temp = Template(prompt_temp)
print(prompt_temp.template)

This is an extractive summarization task. You will be provided with a transcript of a conversation between two participants engaged in a collaborative object matching task. One partcipant is responsible for describing target objects one at a time, while the other participant is responsible for matching the described objects. Your task is to extract the descriptive phrases used by the describer during the conversation for each target object descrived. Extract these phrases verbatim from the excerpts (exclude disfluencies and fillers) and output them in a JSON fomat as follows:

{
    "object_#1": "descriptive phrase for object 1",
    "object_#2": "descriptive phrase for object 2",
    ...
}

Here are two examples to illustrate the format:

Exemplar 1:

{
    "object_#1": "long, handle, narrow",
    "object_#2": "handle, pretty small",
    "object_#3": "shorter handle, full, lot of spaces in between with the weaving",
    "object_#4": "does not have handle, pretty big",
    "object_#5":

In [8]:
dogs_df["object_category"] = "dogs"
baskets_df["object_category"] = "baskets"
combined_df = pd.concat([dogs_df, baskets_df]).reset_index(drop=True)
combined_df.shape, combined_df.columns

((80, 14),
 Index(['Round', 'Order', 'TargetObjectIndex', 'Pair_20', 'Pair_21', 'Pair_22',
        'Pair_23', 'Pair_24', 'Pair_34', 'Pair_35', 'Pair_37', 'Pair_43',
        'Pair_45', 'object_category'],
       dtype='object'))

In [9]:
from src.data.utils import get_conversations

pairs = ['Pair_20', 'Pair_21', 'Pair_22', 'Pair_23', 'Pair_24', 
         'Pair_34', 'Pair_35', 'Pair_37', 'Pair_43', 'Pair_45',]

sub = combined_df[(combined_df['object_category'] == 'dogs')]
FirstRoundCardIndex = sub[sub.Round == 1].TargetObjectIndex
transcript, target_object_indices = get_conversations(sub, 1, "Pair_20", 
                                                      return_entire_transcript=True)
prompt = prompt_temp.substitute(transcript=transcript)
print(prompt)

This is an extractive summarization task. You will be provided with a transcript of a conversation between two participants engaged in a collaborative object matching task. One partcipant is responsible for describing target objects one at a time, while the other participant is responsible for matching the described objects. Your task is to extract the descriptive phrases used by the describer during the conversation for each target object descrived. Extract these phrases verbatim from the excerpts (exclude disfluencies and fillers) and output them in a JSON fomat as follows:

{
    "object_#1": "descriptive phrase for object 1",
    "object_#2": "descriptive phrase for object 2",
    ...
}

Here are two examples to illustrate the format:

Exemplar 1:

{
    "object_#1": "long, handle, narrow",
    "object_#2": "handle, pretty small",
    "object_#3": "shorter handle, full, lot of spaces in between with the weaving",
    "object_#4": "does not have handle, pretty big",
    "object_#5":

### Test

In [32]:
from src.llms.openai import get_response


_, response = get_response(
    prompt,
    model_name="gpt-5-nano"
)
print(response)

{
  "object_#1": "dog standing up very straight, black and brown legs, tail with a little black, black through his torso, basset hound",
  "object_#2": "standing with two front legs straight down, head turned to the side, back legs sticking out to the side, looking down the slope of his back leg",
  "object_#3": "in a seated position",
  "object_#4": "very stout little black dog",
  "object_#5": "poodle",
  "object_#6": "Doberman pinscher",
  "object_#7": "cocker spaniel",
  "object_#8": "looks like a pug, or a miniature boxer",
  "object_#9": "looks like a German shepherd",
  "object_#10": "Yorkie with his tail straight up"
}


### Deployment

In [ ]:
from tqdm import tqdm
from src.llms.openai import get_response


annotated_llm_df_fp = "baskets_dogs_data/transcript_level_entrainment_coding_llm_annotations.csv"

annotated_llm_df = pd.DataFrame(columns=["object_category", "pair_ix", "round_ix", "target_object_indices", "model_response"])

if os.path.exists(annotated_llm_df_fp):
    annotated_llm_df = pd.read_csv(annotated_llm_df_fp)

if len(annotated_llm_df) == 2 * 4 * 10:
    print(f"Loaded existing LLM annotated data with {len(annotated_llm_df)} rows.")

else:
    llm_annotations = []
    p_bar = tqdm(total=2 * 4 * 10)
    
    cols = ["object_category", "pair_ix", "round_ix", "target_object_indices", "model_response"]

    for object_category in ["dogs", "baskets"]:
        combined_df_sub = combined_df[combined_df.object_category == object_category]

        for pair in pairs:
            pair_ix = pair.replace("Pair_", "")

            for i in range(1, 5):
                transcript, target_object_indices = get_conversations(
                    combined_df_sub, i, pair, 
                    return_entire_transcript=True
                )

                if len(
                    annotated_llm_df[
                        (annotated_llm_df.object_category == object_category)
                        & (annotated_llm_df.pair_ix == pair_ix)
                        & (annotated_llm_df.round_ix == i)
                    ]
                ) > 0:
                    p_bar.update(1)
                    continue
                
                prompt = prompt_temp.substitute(transcript=transcript)

                _, response = get_response(
                    prompt,
                    model_name="gpt-5-nano"
                )

                llm_annotations.append({
                    "object_category": object_category,
                    "pair_ix": pair_ix,
                    "round_ix": i,
                    "target_object_indices": target_object_indices,
                    "model_response": response
                })
                p_bar.update(1)
    
            annotated_llm_df = pd.DataFrame(llm_annotations)[cols]
            annotated_llm_df.to_csv(annotated_llm_df_fp, index=False)
    p_bar.close()

annotated_llm_df

Loaded existing LLM annotated data with 80 rows.


,object_category,pair_ix,round_ix,target_object_indices,model_response
0,dogs,20,1,"[2, 8, 4, 10, 6, 3, 7, 1, 9, 5]","{\n ""object_#1"": ""dog standing up very straig..."
1,dogs,20,2,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]","{\n ""object_#1"": ""pushed in with his legs all..."
2,dogs,20,3,"[6, 5, 9, 7, 1, 10, 4, 8, 2, 3]","{\n ""object_#1"": ""poodle"",\n ""object_#2"": ""e..."
3,dogs,20,4,"[5, 9, 1, 7, 3, 6, 10, 4, 8, 2]","{\n ""object_#1"": ""bearded English"",\n ""objec..."
4,dogs,21,1,"[2, 8, 4, 10, 6, 3, 7, 1, 9, 5]","{\n ""object_#1"": ""standing in a really uprigh..."
...,...,...,...,...,...
75,baskets,43,4,"[8, 2, 6, 4, 1, 10, 5, 7, 3, 9]","{\n ""object_#1"": ""long, has a cover and half ..."
76,baskets,45,1,"[9, 3, 7, 5, 10, 1, 4, 6, 2, 8]","{\n ""object_#1"": ""rectangle shaped basket, ha..."
77,baskets,45,2,"[6, 9, 1, 7, 8, 10, 4, 3, 2, 5]","{\n ""object_#1"": ""the heart shaped one"",\n ""..."
78,baskets,45,3,"[10, 8, 2, 4, 6, 5, 7, 3, 9, 1]","{\n ""object_#1"": ""smallest basket with the pi..."


In [13]:
import re
import json


def parse_llm_response(response):
    response_ = re.findall(r"\{[^\}]+\}", response)
    if response:
        response = response_[0]
    
    try:
        parsed = json.loads(response)
    except json.JSONDecodeError:
        parsed = {}

    return parsed


for ix, row in annotated_llm_df.iterrows():
    parsed_response = parse_llm_response(row.model_response)
    annotated_llm_df.at[ix, "parsed_response"] = parsed_response

In [14]:
annotated_llm_df

,object_category,pair_ix,round_ix,target_object_indices,model_response,parsed_response
0,dogs,20,1,"[2, 8, 4, 10, 6, 3, 7, 1, 9, 5]","{\n ""object_#1"": ""dog standing up very straig...","{'object_#1': 'dog standing up very straight, ..."
1,dogs,20,2,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]","{\n ""object_#1"": ""pushed in with his legs all...",{'object_#1': 'pushed in with his legs all the...
2,dogs,20,3,"[6, 5, 9, 7, 1, 10, 4, 8, 2, 3]","{\n ""object_#1"": ""poodle"",\n ""object_#2"": ""e...","{'object_#1': 'poodle', 'object_#2': 'english ..."
3,dogs,20,4,"[5, 9, 1, 7, 3, 6, 10, 4, 8, 2]","{\n ""object_#1"": ""bearded English"",\n ""objec...","{'object_#1': 'bearded English', 'object_#2': ..."
4,dogs,21,1,"[2, 8, 4, 10, 6, 3, 7, 1, 9, 5]","{\n ""object_#1"": ""standing in a really uprigh...",{'object_#1': 'standing in a really upright po...
...,...,...,...,...,...,...
75,baskets,43,4,"[8, 2, 6, 4, 1, 10, 5, 7, 3, 9]","{\n ""object_#1"": ""long, has a cover and half ...","{'object_#1': 'long, has a cover and half rect..."
76,baskets,45,1,"[9, 3, 7, 5, 10, 1, 4, 6, 2, 8]","{\n ""object_#1"": ""rectangle shaped basket, ha...","{'object_#1': 'rectangle shaped basket, handle..."
77,baskets,45,2,"[6, 9, 1, 7, 8, 10, 4, 3, 2, 5]","{\n ""object_#1"": ""the heart shaped one"",\n ""...","{'object_#1': 'the heart shaped one', 'object_..."
78,baskets,45,3,"[10, 8, 2, 4, 6, 5, 7, 3, 9, 1]","{\n ""object_#1"": ""smallest basket with the pi...",{'object_#1': 'smallest basket with the pink r...


In [24]:
from collections import Counter

parsed_response_len = []

for ix, row in annotated_llm_df.iterrows():
    parsed_response = row.parsed_response

    parsed_response_len.append(len(parsed_response))


Counter(parsed_response_len)

Counter({10: 76, 11: 2, 12: 1, 9: 1})

In [27]:
extractions = []
cols = ["object_category", "pair_ix", "FirstRoundCardIndex", 
        "Round1", "Round2", "Round3", "Round4"]


for object_category in annotated_llm_df.object_category.unique():
    for pair_ix in annotated_llm_df.pair_ix.unique():
        df_subset = annotated_llm_df[
            (annotated_llm_df.object_category == object_category)
            & (annotated_llm_df.pair_ix == pair_ix)
        ]
        card_indices_map = dict()
        parsed_responses = dict()

        for round_ix in range(1, len(df_subset) + 1):
            df_subset_round = df_subset[df_subset.round_ix == round_ix]
            parsed_response = df_subset_round.parsed_response.values[0]
            parsed_responses[round_ix] = parsed_response
            card_indices_map[round_ix] = df_subset_round.target_object_indices.values[0]
            
        
        for i in range(len(card_indices_map[1])):
            r1_target_object_ix = card_indices_map[1][i]
            extraction = {
                "object_category": object_category,
                "pair_ix": pair_ix,
                "FirstRoundCardIndex": f"Card{i+1}",
            }

            for round_ix in range(1, len(card_indices_map) + 1):
                round_key = f"Round{round_ix}"
                target_object_ix = card_indices_map[round_ix].index(r1_target_object_ix) + 1
                object_key = f"object_#{target_object_ix}"

                if round_ix == 1:
                    print(i, object_key)
                
                descriptive_phrase = parsed_responses[round_ix].get(object_key, "")
                extraction[round_key] = descriptive_phrase
            extractions.append(extraction)

extracted_df = pd.DataFrame(extractions)[cols]
extracted_df

0 object_#1
1 object_#2
2 object_#3
3 object_#4
4 object_#5
5 object_#3
6 object_#4
7 object_#8
8 object_#3
9 object_#4
10 object_#11
11 object_#12
12 object_#3
13 object_#4
14 object_#15
15 object_#3
16 object_#4
17 object_#18
18 object_#3
19 object_#4
20 object_#21
21 object_#3
22 object_#4
23 object_#11
24 object_#3
25 object_#4
26 object_#27
27 object_#3
28 object_#4
29 object_#30
30 object_#31
0 object_#1
1 object_#2
2 object_#3
3 object_#4
4 object_#5
5 object_#3
6 object_#4
7 object_#8
8 object_#3
9 object_#4
10 object_#11
11 object_#12
12 object_#3
13 object_#4
14 object_#15
15 object_#3
16 object_#4
17 object_#18
18 object_#3
19 object_#4
20 object_#21
21 object_#3
22 object_#4
23 object_#11
24 object_#3
25 object_#4
26 object_#27
27 object_#3
28 object_#4
29 object_#30
30 object_#31
0 object_#1
1 object_#2
2 object_#3
3 object_#4
4 object_#5
5 object_#3
6 object_#4
7 object_#8
8 object_#3
9 object_#4
10 object_#11
11 object_#12
12 object_#3
13 object_#4
14 object_#15
15 objec

,object_category,pair_ix,FirstRoundCardIndex,Round1,Round2,Round3,Round4
0,dogs,20,Card1,"dog standing up very straight, black and brown...",pushed in with his legs all the way down,poodle,bearded English
1,dogs,20,Card2,standing with his two front legs straight down...,our bearded English dog,,
2,dogs,20,Card3,"seated position, sitting, head turned",the Doberman,"husky, or the sled dog",pug
3,dogs,20,Card4,"very stout little black dog, very short, curly...",the brown dog that's sitting,cocker spaniel,cocker spaniel
4,dogs,20,Card5,poodle,,,
...,...,...,...,...,...,...,...
615,baskets,45,Card27,,,rectangle shaped one with the handle in the mi...,the one with the green background with triangl...
616,baskets,45,Card28,"circular one, crisscross box baskets, round ha...","no lid, no handle, triangles on the bowl, gold...",goldish brown with two small handles on the si...,the heart shaped one
617,baskets,45,Card29,"no lid, wide circle, tiny sewing things, wides...",circular one with the crisscrosses and the lit...,heart shaped,the one that has the two small handles and the...
618,baskets,45,Card30,,,"the most circular and widest, with no lid and ...",the circular one with the square handle and th...


In [16]:
df = pd.read_csv("baskets_dogs_data/entrainment_coding_llm_annotations.csv")
df

,object_category,pair_ix,FirstRoundCardIndex,Model_Response,Round1,Round2,Round3,Round4
0,dogs,20,Card1,"```json\n{\n ""Round1"": ""dog standing up ver...","dog standing up very straight, black and brown...","large basset hound, black and the brown legs, ...","the basset hound, big guy",the big basset hound
1,dogs,20,Card2,"{\n ""Round1"": ""standing, uh, with his two f...","standing, uh, with his two front legs, uh...st...",dog that looks like he's looking down his leg,"guy looking down his leg, The *schnauzer*","The schnauzer, looking down his leg"
2,dogs,20,Card3,"{\n ""Round1"": ""seated position, sitting, he...","seated position, sitting, head turned",brown dog that's sitting,sitting dog,"dog that's sitting, turned to the back"
3,dogs,20,Card4,"{\n ""Round1"": ""stout little, uh, black dog,...","stout little, uh, black dog, very short, littl...",weiner dog,weiner dog,weiner
4,dogs,20,Card5,"{\n ""Round1"": ""the poodle"",\n ""Round2"": ...",the poodle,the black poodle,the poodle,The poodle
...,...,...,...,...,...,...,...,...
195,baskets,45,Card6,"{\n ""Round1"": ""light goldish basket, has li...","light goldish basket, has like triangles, gree...","no lid, no handle, triangles on the bowl, gold...","green background and triangles, goldish brown","green background with triangles, goldish brown"
196,baskets,45,Card7,"{\n ""Round1"": ""has a lid and um it's like g...",has a lid and um it's like goldish and it has ...,"is the one with the lid, It's goldish brown wi...",the one with the two ha--it's goldish brown wi...,the one that has the two small handles and the...
197,baskets,45,Card8,"{\n ""Round1"": ""triangle…shaped basket"",\n ...",triangle…shaped basket,heart shaped,heart shaped,heart shaped
198,baskets,45,Card9,"```json\n{\n ""Round1"": ""circular one, has a...","circular one, has a handle in the middle",round one with the lid and a handle,"roundish with the flat lid and a handle, squar...",circular one with the square handle and the fl...


In [30]:
# annotated_llm_df.to_csv("baskets_dogs_data/entrainment_coding_llm_annotations.csv", index=False)

### Evaluation

In [41]:
from rouge import Rouge
from sentence_transformers import SentenceTransformer, util

def rouge_f1(system_summary: str, reference_summary: str):
    rouge = Rouge()
    scores = rouge.get_scores(system_summary, reference_summary)[0]

    r1_f1 = scores["rouge-1"]["f"]
    r2_f1 = scores["rouge-2"]["f"]
    rL_f1 = scores["rouge-l"]["f"]

    return r1_f1, r2_f1, rL_f1


# Load once (global) so you don't re-load the model on every call
_sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

def sbert_score(text1: str, text2: str) -> float:
    """
    Compute SBERT cosine similarity between two strings.
    Returns a single float (typically in [0, 1] for normal sentences).
    """
    emb1 = _sbert_model.encode(text1, convert_to_tensor=True)
    emb2 = _sbert_model.encode(text2, convert_to_tensor=True)

    # util.cos_sim returns a 1x1 tensor here
    score = util.cos_sim(emb1, emb2).item()
    return float(score)

/home/jack/anaconda3/envs/LangCogVis/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [35]:
test_df.head(1)

,object_category,pair_ix,split,FirstRoundCardIndex,Round1,Round2,Round3,Round4
0,baskets,45,test,Card1,"rectangle shaped, handle in middle, no lid, empty","rectangle shaped with handle in middle, no lid",rectangle shaped with handle in middle,rectangular shaped with handle in middle


In [36]:
annotated_llm_df.head(1)

,object_category,pair_ix,FirstRoundCardIndex,Model_Response,Round1,Round2,Round3,Round4
0,dogs,20,Card1,"```json\n{\n ""Round1"": ""dog standing up ver...","dog standing up very straight, black and brown...","large basset hound, black and the brown legs, ...","the basset hound, big guy",the big basset hound


In [42]:
evaluation_results = []

for object_category in test_df.object_category.unique():
    test_df_sub = test_df[test_df.object_category == object_category]
    annotated_llm_df_sub = annotated_llm_df[annotated_llm_df.object_category == object_category]

    for pair_ix in test_df_sub.pair_ix.unique():
        test_df_subsub = test_df_sub[test_df_sub.pair_ix == pair_ix]
        annotated_llm_df_subsub = annotated_llm_df_sub[annotated_llm_df_sub.pair_ix == pair_ix]

        for ix, row in test_df_subsub.iterrows():
            FirstRoundCardIndex = row.FirstRoundCardIndex
            annotated_row = annotated_llm_df_subsub[
                annotated_llm_df_subsub.FirstRoundCardIndex == FirstRoundCardIndex
            ].iloc[0]

            for round_num in range(1, 5):
                reference_summary = row[f"Round{round_num}"]
                system_summary = annotated_row[f"Round{round_num}"]

                r1_f1, r2_f1, rL_f1 = rouge_f1(system_summary, reference_summary)

                evaluation_results.append({
                    "object_category": object_category,
                    "pair_ix": pair_ix,
                    "FirstRoundCardIndex": FirstRoundCardIndex,
                    "round_num": round_num,
                    "rouge-1_f1": r1_f1,
                    "rouge-2_f1": r2_f1,
                    "rouge-L_f1": rL_f1,
                    "sbert_score": sbert_score(system_summary, reference_summary),
                })

evaluation_df = pd.DataFrame(evaluation_results)
evaluation_df

,object_category,pair_ix,FirstRoundCardIndex,round_num,rouge-1_f1,rouge-2_f1,rouge-L_f1,sbert_score
0,baskets,45,Card1,1,0.400000,0.142857,0.400000,0.690133
1,baskets,45,Card1,2,0.625000,0.266667,0.625000,0.785143
2,baskets,45,Card1,3,0.857143,0.307692,0.857143,0.944746
3,baskets,45,Card1,4,0.923077,0.500000,0.923077,0.981395
4,baskets,45,Card2,1,0.666667,0.360000,0.666667,0.878465
...,...,...,...,...,...,...,...,...
795,dogs,24,Card9,4,0.800000,0.750000,0.800000,0.908934
796,dogs,24,Card10,1,0.842105,0.631579,0.842105,0.891194
797,dogs,24,Card10,2,0.761905,0.604651,0.761905,0.906219
798,dogs,24,Card10,3,0.933333,0.714286,0.933333,0.976210


In [43]:
evaluation_df.groupby("round_num")[["rouge-1_f1", "rouge-2_f1", "rouge-L_f1", "sbert_score"]].describe().T

round_num                   1           2           3           4
rouge-1_f1  count  200.000000  200.000000  200.000000  200.000000
            mean     0.620296    0.656635    0.652695    0.693043
            std      0.206122    0.226837    0.222252    0.220746
            min      0.000000    0.000000    0.000000    0.000000
            25%      0.500000    0.542424    0.500000    0.553030
            50%      0.635255    0.700000    0.666667    0.727273
            75%      0.770105    0.818182    0.823529    0.845865
            max      1.000000    1.000000    1.000000    1.000000
rouge-2_f1  count  200.000000  200.000000  200.000000  200.000000
            mean     0.382353    0.414858    0.400994    0.417823
            std      0.234339    0.274800    0.303107    0.311215
            min      0.000000    0.000000    0.000000    0.000000
            25%      0.209619    0.222222    0.153846    0.142857
            50%      0.356471    0.418860    0.400000    0.428571
            75%      0.533333    0.600000    0.600000    0.626645
            max      1.000000    1.000000    1.000000    1.000000
rouge-L_f1  count  200.000000  200.000000  200.000000  200.000000
            mean     0.606738    0.653430    0.648922    0.689177
            std      0.211315    0.230507    0.225017    0.225425
            min      0.000000    0.000000    0.000000    0.000000
            25%      0.481117    0.533333    0.500000    0.553030
            50%      0.621820    0.700000    0.666667    0.727273
            75%      0.760142    0.818182    0.819519    0.845865
            max      1.000000    1.000000    1.000000    1.000000
sbert_score count  200.000000  200.000000  200.000000  200.000000
            mean     0.823202    0.824418    0.853600    0.864320
            std      0.121203    0.136497    0.112020    0.122698
            min      0.334573    0.450227    0.461105    0.429996
            25%      0.749782    0.723836    0.772793    0.797745
            50%      0.842312    0.844328    0.868152    0.890433
            75%      0.920822    0.943523    0.953529    0.971339
            max      1.000000    1.000000    1.000000    1.000000

In [44]:
evaluation_df.to_csv("notebooks/outputs/llm_entrainment_coding_evaluation.csv", index=False)